In [56]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [57]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [58]:
def mutation_count_frequency(seqs, redux, mut_pair, J, min_pos, max_pos):
    pair1,pair2 = functions.split_pairs(mut_pair)
    reduced_pair1 = functions.unreduced_to_reduced(redux, pair1)
    reduced_pair2 = functions.unreduced_to_reduced(redux, pair2)
    # print(f"Reduced pair 1: {reduced_pair1}")
    # print(f"Reduced pair 2: {reduced_pair2}")
    wt1, pos1, mt1 = functions.split_pair(reduced_pair1)
    wt2, pos2, mt2 = functions.split_pair(reduced_pair2)

    count_withDMC = 0
    count_total = 0
    for seq in seqs:
        # de1 = functions.calculate_delta_e(reduced_pair1, seq, J, min_pos, max_pos)
        # de2 = functions.calculate_delta_e(reduced_pair2, seq, J, min_pos, max_pos)
        # de12 = functions.calculate_delta_e_double(reduced_pair1, reduced_pair2, seq, J, min_pos, max_pos)

        count_total += 1
        if seq[pos1-min_pos] == mt1 and seq[pos2-min_pos] == mt2:
            count_withDMC += 1
    
    if count_withDMC <= 4:
        print(f" pair: {mut_pair}, count: {count_withDMC}")
        # print(f"Total sequences: {len(seqs)}")
        # print(f"Total mutations with DMC: {count_withDMC}")
        # print(f"Frequency of DMC: {count_withDMC / count_total:.4f}")
    return count_withDMC, count_total, count_withDMC / count_total

In [59]:
import csv

IN_pairs = [
    'G140S-Q148H', 'Y143C-S230R', 'G140A-Q148K', 'G140S-Q148R', 'G140S-Q148K',
    'G140A-Q148R', 'E138K-Q148K', 'G140A-Q148H', 'E138K-Q148R', 'Y143C-S230K',
    'N155H-E170A', 'E138K-S147G', 'S147G-Q148R', 'Y143R-V151V', 'E138K-Q148H',
    'S147G-L158V', 'E92Q-K215R', 'E138A-Q148H', 'E138K-S230K', 'E138K-Y194C',
    'Y143R-N155H', 'Q148H-N155H', 'G140S-N155H', 'G140S-S147G', 'Y143R-Q148H',
    'Q148R-N155H', 'G140S-Y143R', 'E92Q-Y143R', 'G140A-S147G', 'E92Q-Q148H',
    'N155H-G193D', 'E92Q-G140S', 'Y143R-Q148R', 'G140A-S230N', 'S147G-V151V',
    'Q148R-V165I', 'N155H-V176L', 'N155H-K160R', 'Y143C-L234I', 'E92Q-G163R'
]

in_mutation_rows = []

for pair in IN_pairs:
    # print(f"Analyzing pair: {pair}")
    count_withDMC, count_total, observed_freq = mutation_count_frequency(
        IN_all_seq, IN_redux, pair, IN_J, 1, 263
    )

    in_mutation_rows.append({
        'mutation pair': pair,
        'count_total': count_total,
        'count_with_mutation': count_withDMC,
        'observed_frequency': observed_freq
    })

out_path = 'IN/data/in_DM_freq_summary.csv'
with open(out_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=['mutation pair', 'count_total', 'count_with_mutation', 'observed_frequency']
    )
    writer.writeheader()
    writer.writerows(in_mutation_rows)
    zero_count_pairs = [row['mutation pair'] for row in in_mutation_rows if row['count_with_mutation'] == 0]

    print("Pairs with count_with_mutation = 0:")
    for p in zero_count_pairs:
        print(p)
print(f"Saved {len(in_mutation_rows)} rows to {out_path}")

 pair: G140A-Q148K, count: 4
 pair: G140S-Q148K, count: 2
 pair: E138K-Q148K, count: 4
 pair: G140A-Q148H, count: 0
 pair: Y143C-S230K, count: 1
 pair: S147G-L158V, count: 1
 pair: E92Q-K215R, count: 1
 pair: E138K-S230K, count: 1
 pair: E138K-Y194C, count: 2
 pair: Y143R-N155H, count: 1
 pair: Q148H-N155H, count: 1
 pair: G140S-N155H, count: 1
 pair: G140S-S147G, count: 0
 pair: Y143R-Q148H, count: 0
 pair: Q148R-N155H, count: 1
 pair: G140S-Y143R, count: 0
 pair: E92Q-Y143R, count: 0
 pair: G140A-S147G, count: 0
 pair: E92Q-Q148H, count: 0
 pair: N155H-G193D, count: 0
 pair: E92Q-G140S, count: 0
 pair: Y143R-Q148R, count: 0
 pair: G140A-S230N, count: 0
 pair: Q148R-V165I, count: 0
 pair: N155H-V176L, count: 0
 pair: N155H-K160R, count: 0
 pair: Y143C-L234I, count: 0
 pair: E92Q-G163R, count: 0
Pairs with count_with_mutation = 0:
G140A-Q148H
G140S-S147G
Y143R-Q148H
G140S-Y143R
E92Q-Y143R
G140A-S147G
E92Q-Q148H
N155H-G193D
E92Q-G140S
Y143R-Q148R
G140A-S230N
Q148R-V165I
N155H-V176L
N155

In [60]:
import csv

PR_weights_path = 'PR/data/pr.exper.weights.txt'
len_PR_all_seqs = len(PR_all_seq)
with open(PR_weights_path, 'r') as f:
    PR_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

PR_pairs = [
    'D30N-N88D',
    'V32I-I47V',
    'G48V-I54A',
    'D30N-K45Q',
    'I54A-V82A',
    'I54V-V82A',
    'M46I-L76V',
    'I54V-V82T',
    'I54A-V82T',
    'G48V-V82A',
    'I54A-A71I',
    'M46I-N88T',
    'L90M-C95F',
    'V32I-M46I',
    'G48V-V82T',
    'I54V-T91S',
    'M46I-F53Y',
    'M46L-K55R',
    'M46L-V82A',
    'M46I-K55R',
    #antagonisitc pairs
    'I50V-I84V',
    'V82A-I84V',
    'V32I-I54V',
    'V32I-I54A',
    'G48V-G73S',
    'D30N-V82A',
    'D30N-V82T',
    'D30N-G73S',
    'D30N-L90M',
    'V82A-N88S',
    'V82T-N83D',
    'V32I-L76V',
    'I54V-N88S',
    'M46I-G48V',
    'D30N-V82I',
    'G48V-G73T',
    'I50V-I54L',
    'N88S-L90M',
    'L76V-N88D',
    'V82T-N88D',
]
for pair in PR_pairs:
    # print(f"Analyzing pair: {pair}")
    count_withDMC, count_total, observed_freq = mutation_count_frequency(PR_all_seq, PR_redux, pair, PR_J, 1, 99)

    if pair == PR_pairs[0]:
        pr_mutation_rows = []

    pr_mutation_rows.append({
        'mutation pair': pair,
        'count_total': count_total,
        'count_with_mutation': count_withDMC,
        'observed_frequency': observed_freq
    })

    if pair == PR_pairs[-1]:
        out_path = 'PR/data/pr_DM_freq_summary.csv'
        with open(out_path, 'w', newline='') as csvfile:
            writer = csv.DictWriter(
                csvfile,
                fieldnames=['mutation pair', 'count_total', 'count_with_mutation', 'observed_frequency']
            )
            writer.writeheader()
            writer.writerows(pr_mutation_rows)
        zero_count_pairs = [row['mutation pair'] for row in pr_mutation_rows if row['count_with_mutation'] == 0]

        print("Pairs with count_with_mutation = 0:")
        for p in zero_count_pairs:
            print(p)
        print(f"Saved {len(pr_mutation_rows)} rows to {out_path}")


 pair: I50V-I84V, count: 0
 pair: V32I-I54A, count: 3
 pair: G48V-G73S, count: 3
 pair: D30N-V82T, count: 1
 pair: D30N-G73S, count: 1
 pair: V82A-N88S, count: 1
 pair: V82T-N83D, count: 0
 pair: I54V-N88S, count: 1
 pair: D30N-V82I, count: 0
 pair: G48V-G73T, count: 0
 pair: I50V-I54L, count: 0
 pair: L76V-N88D, count: 0
 pair: V82T-N88D, count: 3
Pairs with count_with_mutation = 0:
I50V-I84V
V82T-N83D
D30N-V82I
G48V-G73T
I50V-I54L
L76V-N88D
Saved 40 rows to PR/data/pr_DM_freq_summary.csv


In [61]:
RT_pairs = [
    'K101E-G190S', 'K101E-G190A', 'K103N-P225H', 'L100I-K103N', 'K101P-K103S',
    'Y181C-H221Y', 'K103S-G190A', 'K103S-P225H', 'L100I-K103R', 'V108I-H221Y',
    'K103S-D192N', 'L100I-K103S', 'K101E-E138A', 'Y181C-G190A', 'K103S-D177N',
    'V108I-V189I', 'K101E-E138K', 'E138A-G190E', 'K101P-D192N', 'V108I-L109V',
    'L100I-Y181C', 'K103N-V179D', 'V108I-Y188L', 'K103N-G190S', 'K103N-G190E',
    'L100I-G190A', 'K101E-K103N', 'L100I-K101P', 'E138A-Y181C', 'L100I-V108I',
    'H221Y-K223Q', 'K103S-Y188L', 'Y181C-Y188L', 'K101E-K103S', 'Y188L-G190S',
    'V108I-G190S', 'K101P-Y181C', 'K103S-Y181C', 'L100I-V106I', 'Y181C-P225H'
]

rt_mutation_rows = []

for pair in RT_pairs:
    pair = pair.replace(' -', '-').replace('- ', '-').strip()
    # print(f"Analyzing pair: {pair}")
    count_withDMC, count_total, observed_freq = mutation_count_frequency(
        RT_all_seq, RT_redux, pair, RT_J, 39, 226
    )

    rt_mutation_rows.append({
        'mutation pair': pair,
        'count_total': count_total,
        'count_with_mutation': count_withDMC,
        'observed_frequency': observed_freq
    })

out_path = 'RT/data/rt_NNRTI_DM_freq_summary.csv'
with open(out_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=['mutation pair', 'count_total', 'count_with_mutation', 'observed_frequency']
    )
    writer.writeheader()
    writer.writerows(rt_mutation_rows)

print(f"Saved {len(rt_mutation_rows)} rows to {out_path}")

 pair: L100I-K101P, count: 1
 pair: K103S-Y188L, count: 3
Saved 40 rows to RT/data/rt_NNRTI_DM_freq_summary.csv


In [62]:
rt_nrti_pairs = [
    'F116Y -Q151M', 'M41L-T215Y', 'V75I-I132L', 'K70R-K219E', 'D67N-K219Q',
    'K70R-K219Q', 'L210W-T215Y', 'F116Y -Q151L', 'K65R-S68N', 'V75I-F77L',
    'M41L-T215F', 'D67N-K219E', 'L210W-T215S', 'M41L-T215S', 'D67N-K70R',
    'L74V -Y115F', 'L74V -L100I', 'A62V -V75I', 'F116Y -Q151R', 'A62V -V75T',
    'K65R-T215Y', 'D67N-S68G', 'L74V -V75M', 'L210W-F214L', 'K65R-T215F',
    'L74V -F77L', 'M41L-K46Q', 'L74V -V75I', 'L74V -F116Y', 'L210W-K219Q',
    'K70R-L210W', 'L74I-Q151M', 'M41L-F214L', 'L74I-F116Y', 'D67N-S68R',
    'L74I-V75M', 'K70R-L210S', 'K65R-D67N', 'L74V -Q151M', 'M184V -Y188C'
]

rt_nrti_mutation_rows = []

for pair in rt_nrti_pairs:
    pair = pair.replace(' -', '-').replace('- ', '-').strip()
    count_withDMC, count_total, observed_freq = mutation_count_frequency(
        RT_all_seq, RT_redux, pair, RT_J, 39, 226
    )

    rt_nrti_mutation_rows.append({
        'mutation pair': pair,
        'count_total': count_total,
        'count_with_mutation': count_withDMC,
        'observed_frequency': observed_freq
    })

out_path = 'RT/data/rt_NRTI_DM_freq_summary.csv'
with open(out_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=['mutation pair', 'count_total', 'count_with_mutation', 'observed_frequency']
    )
    writer.writeheader()
    writer.writerows(rt_nrti_mutation_rows)

print(f"Saved {len(rt_nrti_mutation_rows)} rows to {out_path}")

 pair: F116Y-Q151R, count: 2
 pair: M41L-K46Q, count: 2
 pair: L74I-Q151M, count: 4
 pair: L74I-F116Y, count: 2
Saved 40 rows to RT/data/rt_NRTI_DM_freq_summary.csv
